# Building Tools for Agents

*Notebook 02*

Give your agent real capabilities by turning Python functions into tools.


---

## 🔧 Setup

In [ ]:
from datetime import datetime
from pathlib import Path
from dotenv import load_dotenv
load_dotenv(dotenv_path=Path("..") / ".env")

from agents import Agent, Runner, function_tool

MODEL = "gpt-5-mini"  # Course default, fast and cost-effective for most tasks

print("✅ Ready!")

---

## 🛠️ Part 1: Your First Tool

Without tools, an agent can reason from the conversation and its model knowledge.

It cannot query your private or live systems.

Tools let it call Python functions connected to databases, files, and APIs.

### Before: Agent without tools

In [ ]:
message = "What department does employee E003 work in?"

agent_no_tools = Agent(
    name="NoToolsAgent",
    instructions="You are a helpful assistant.",
    model=MODEL
)

result = await Runner.run(agent_no_tools, input=message)

print(result.final_output)

### After: Agent with an employee lookup tool

In [ ]:
# Simulated employee database
EMPLOYEES = {
    "E001": {"name": "Sarah Chen", "department": "Engineering", "salary": 95000},
    "E002": {"name": "Marcus Johnson", "department": "Marketing", "salary": 78000},
    "E003": {"name": "Priya Patel", "department": "Finance", "salary": 88000},
    "E004": {"name": "Tom Rivera", "department": "Engineering", "salary": 102000},
    "E005": {"name": "Lisa Wong", "department": "HR", "salary": 72000},
}

@function_tool
def get_employee(employee_id: str) -> str:
    """Look up an employee's name, department, and salary by their employee ID."""
    print(f"  🔧 [tool] get_employee called with {employee_id}")
    employee = EMPLOYEES.get(employee_id.upper())
    if employee:
        return (
            f"Name: {employee['name']}, "
            f"Department: {employee['department']}, "
            f"Salary: ${employee['salary']:,}"
        )
    return f"No employee found with ID {employee_id}"

# --------------------------------------------------------------
print("✅ get_employee() ready")

The `@function_tool` decorator turns any Python function into an agent tool:

- **Name**: tells the agent what the tool is called

- **Docstring**: tells the agent what the tool does and when to use it

- **Type hints**: define the inputs and generate the schema (no manual JSON)

#### Run Agent with Employee Lookup Tool

In [ ]:
agent_with_tools = Agent(
    name="EmployeeAgent",
    instructions="You are a helpful assistant.",
    model=MODEL,
    tools=[get_employee]
)

result = await Runner.run(agent_with_tools, input=message)

print(result.final_output)

The tool marker confirms the call and argument.

Open the [Traces dashboard](https://platform.openai.com/traces) to see how the result shaped the answer.

---

## 🗓️ Part 2: A Tool with No Parameters

Not all tools take input.

Some just return live information: the current time, system status, environment data.

In [ ]:
@function_tool
def get_current_datetime() -> str:
    """Return the current date and time."""
    print("  🔧 [tool] get_current_datetime called")
    return datetime.now().strftime("%A, %B %d, %Y at %I:%M %p")

# --------------------------------------------------------------
print("✅ get_current_datetime() ready")

#### Run Agent with DateTime Tool

In [ ]:
agent_datetime = Agent(
    name="DateTimeAgent",
    instructions="Use the get_current_datetime tool for all time and date questions.",
    model=MODEL,
    tools=[get_current_datetime]
)

result = await Runner.run(agent_datetime, input="What day of the week is it today?")

print(result.final_output)

---

## 🧠 Part 3: How the Agent Decides

An agent doesn't use every tool on every run.

The model sees the request and its instructions.

It also sees each tool's name, description, and parameter schema.

Clear docstrings make the right choice more likely.

In [ ]:
# An agent with both tools, watch it choose
agent_multi = Agent(
    name="MultiToolAgent",
    instructions="You are a helpful assistant. Use tools whenever they are relevant.",
    model=MODEL,
    tools=[get_employee, get_current_datetime]
)

# --------------------------------------------------------------
print("✅ MultiToolAgent ready")

#### Test: Employee Question

In [ ]:
result = await Runner.run(agent_multi, input="What is employee E001's salary?")

print(result.final_output)

#### Test: Time Question

In [ ]:
result = await Runner.run(agent_multi, input="What time is it right now?")

print(result.final_output)

#### Test: No Tool Needed

In [ ]:
result = await Runner.run(agent_multi, input="What is the capital of France?")

print(result.final_output)

### 💡 Writing a Good Tool Docstring

A docstring the agent can act on states three things:

1. **What it returns**: the specific data the tool gives back

2. **When to use it**: the trigger condition ("use this when the user asks about…")

3. **What each input means**: if it's not obvious from the parameter name

`"Get employee info"` is too vague.

A stronger docstring names the returned fields and when to use the tool.

---

## 💪 Practice Exercises

### Exercise 1: Word Counter Tool

*Covers: `@function_tool`: building and wiring custom tools*

Create a tool that counts the words in a string, then build an agent that uses it.

In [ ]:
# --------------------------------------------------------------
# 💪 Exercise 1: Word Counter Tool
# --------------------------------------------------------------
# Objective: Build a tool that counts words and an agent that uses it.

# TODO 1: Define a @function_tool called count_words
#             - Parameter: text (str)
#             - Returns: str with the word count
#             - Print a marker when it runs, e.g. "🔧 [tool] count_words called"
#             - Hint: len(text.split()) gives you the word count

# TODO 2: Create an Agent instructed to ALWAYS use count_words for word-count questions

# TODO 3: Run the agent with: "How many words are in this sentence: The quick brown fox"
#             Confirm the tool marker appears in the output

# --- Write your code below this line ---

### Exercise 2: Temperature Converter

*Covers: `@function_tool`: tool inputs, outputs, and docstrings*

Create a tool that converts Celsius to Fahrenheit, then build an agent that uses it.

In [ ]:
# --------------------------------------------------------------
# 💪 Exercise 2: Temperature Converter
# --------------------------------------------------------------
# Objective: Build a temperature conversion tool and an agent that uses it.

# TODO 1: Define a @function_tool called celsius_to_fahrenheit
#             - Parameter: celsius (float)
#             - Formula: (celsius * 9/5) + 32
#             - Returns: str with the result
#             - Print a marker when it runs, e.g. "🔧 [tool] celsius_to_fahrenheit called"

# TODO 2: Create an Agent instructed to ALWAYS use the tool for temperature conversions

# TODO 3: Run the agent with: "What is 100 degrees Celsius in Fahrenheit?"
#             Confirm the tool marker appears in the output

# --- Write your code below this line ---

---

## 🎯 Key Takeaways

**`@function_tool` turns a Python function into an agent capability:**

- The docstring becomes the tool description

- Type hints generate the parameter schema

- You don't write the JSON schema by hand
<br>
<br>

**The model chooses which tool to call:**

- It sees each tool's name, description, and parameter schema

- Clear docstrings make the right call more likely, not guaranteed

---

## 📍 Next Step

**Notebook 03: Writing Effective Agent Instructions**  

Learn to write system instructions that shape how your agent behaves, responds, and uses tools.

---

##### 🔧 [Troubleshooting Guide](https://github.com/barrettscott/openai-agents/blob/main/TROUBLESHOOTING.md#lesson-02-building-tools-for-agents)

---